In [ ]:
import numpy as np
import chaospy as cp
import matplotlib.pyplot as plt
import pyvista as pv

# ---------------------------
# 1. Load Simulation Data
# ---------------------------
# Replace these file names with the paths to your actual text files.
# Here we assume the text files contain 1D arrays (e.g., average values) of pressure and velocity.
pressure_model_noncoarc = np.loadtxt("/svFSI_Inflate/aorta_cmm/48-procs/B_CM_Pressure_average.txt")  # Non-coarctation model
velocity_model_noncoarc = np.loadtxt("/svFSI_Inflate/aorta_cmm/48-procs/B_CM_Velocity_flux.txt")
pressure_model_coarc = np.loadtxt("pressure_coarc.txt")        # Coarctation model
velocity_model_coarc = np.loadtxt("velocity_coarc.txt")

# For demonstration, we take the mean values from each text file as the simulation outputs.
# In practice, you might want to work with field distributions.
sim_pressure_noncoarc = np.mean(pressure_model_noncoarc)
sim_velocity_noncoarc = np.mean(velocity_model_noncoarc)
sim_pressure_coarc = np.mean(pressure_model_coarc)
sim_velocity_coarc = np.mean(velocity_model_coarc)

# ---------------------------
# 2. Read Geometry Files
# ---------------------------
# Here, we load and optionally visualize the VTU/VTP model geometries.
model_noncoarc_geo = pv.read("model_noncoarc.vtu")
model_coarc_geo    = pv.read("model_coarc.vtp")

# Visualize the geometries (pop-up a window for each model)
model_noncoarc_geo.plot(title="Non-coarctation Geometry")
model_coarc_geo.plot(title="Coarctation Geometry")

# ---------------------------
# 3. Set Up the PCE Framework
# ---------------------------
# Assume a single uncertain parameter, xi, representing the “coarctation severity”
# xi = 0 corresponds to the non-coarctation model and xi = 1 corresponds to the coarctation model.
distribution = cp.Uniform(0, 1)  # Uniform distribution over [0,1]
order = 2  # Polynomial order
poly_expansion = cp.orth_ttr(order, distribution)

# Generate quadrature points and weights (we need as many simulation outputs as quadrature points)
# Here we choose a Gaussian quadrature rule.
quad_points, weights = cp.generate_quadrature(order + 1, distribution, rule="G")

# Define a “simulation” function that interpolates between your two models.
# In practice, you might have simulations for intermediate values.
def simulation_response(x, field="pressure"):
    if field == "pressure":
        # Linear interpolation between the two models based on x
        return (1 - x) * sim_pressure_noncoarc + x * sim_pressure_coarc
    elif field == "velocity":
        return (1 - x) * sim_velocity_noncoarc + x * sim_velocity_coarc
    else:
        raise ValueError("Field must be either 'pressure' or 'velocity'")

# Evaluate the simulation response at the quadrature points
pressure_samples = np.array([simulation_response(x, field="pressure") for x in quad_points])
velocity_samples = np.array([simulation_response(x, field="velocity") for x in quad_points])

# ---------------------------
# 4. Compute PCE Coefficients
# ---------------------------
# Use projection via quadrature to compute the expansion coefficients.
pce_pressure_coeffs = cp.fit_quadrature(
    poly_expansion, quad_points, weights, pressure_samples
)
pce_velocity_coeffs = cp.fit_quadrature(
    poly_expansion, quad_points, weights, velocity_samples
)

# Build callable PCE models for pressure and velocity
pce_pressure_model = cp.callable_from_expansion(pce_pressure_coeffs, poly_expansion)
pce_velocity_model = cp.callable_from_expansion(pce_velocity_coeffs, poly_expansion)

# ---------------------------
# 5. Statistical Analysis
# ---------------------------
# Compute mean and variance based on the PCE model
mean_pressure = cp.E(pce_pressure_model, distribution)
var_pressure = cp.Var(pce_pressure_model, distribution)
mean_velocity = cp.E(pce_velocity_model, distribution)
var_velocity = cp.Var(pce_velocity_model, distribution)

print("Pressure Mean:", mean_pressure)
print("Pressure Variance:", var_pressure)
print("Velocity Mean:", mean_velocity)
print("Velocity Variance:", var_velocity)

# Sensitivity Analysis: compute Sobol main sensitivity indices.
sobol_pressure = cp.Sens_m(pce_pressure_coeffs, poly_expansion)
sobol_velocity = cp.Sens_m(pce_velocity_coeffs, poly_expansion)

print("Pressure Sobol Indices:", sobol_pressure)
print("Velocity Sobol Indices:", sobol_velocity)

# ---------------------------
# 6. Visualization of PCE Models
# ---------------------------
# Plot the PCE approximations against the uncertain parameter xi.
xi_vals = np.linspace(0, 1, 100)
pressure_vals = pce_pressure_model(xi_vals)
velocity_vals = pce_velocity_model(xi_vals)

plt.figure()
plt.plot(xi_vals, pressure_vals, label="Pressure PCE Model")
plt.xlabel("Uncertain Parameter xi")
plt.ylabel("Pressure")
plt.title("PCE Pressure Response")
plt.legend()

plt.figure()
plt.plot(xi_vals, velocity_vals, label="Velocity PCE Model")
plt.xlabel("Uncertain Parameter xi")
plt.ylabel("Velocity")
plt.title("PCE Velocity Response")
plt.legend()

plt.show()
